# An autocorrelation problem related to difference bases

In [ ]:
#@title Verification code

# pylint: disable=unused-import
# pylint: disable=g-bad-import-order


linprog = optimize.linprog



def score_function(fun: Callable[[float], float]) -> float:
  """Scores a function."""
  try:
    rhs = integrate.quad(fun, -0.25, 0.25)[0]
  except ZeroDivisionError:
    logging.info('ZeroDivisionError')
    return np.inf, 'ZeroDivisionError', '?'
  rhs = rhs**2
  if rhs < 0.01:
    logging.info('rhs is too small: %s', rhs)
    return np.inf, 'rhs is too small', '?'

  def lhs_inner(x, t):
    return fun(x) * fun(t - x)

  def lhs_integral(t):
    return integrate.quad(lhs_inner, -np.inf, np.inf, args=(t))[0]

  vec_lhs_integral = np.vectorize(lhs_integral)
  try:
    lhs_values = vec_lhs_integral(np.arange(-0.5, 0.5, 0.01))
  except ZeroDivisionError:
    logging.info('ZeroDivisionError')
    return np.inf, rhs, 'ZeroDivisionError'
  lhs_max_value = np.max(lhs_values)
  return lhs_max_value / rhs, lhs_max_value, rhs


def score_all_function_shifts(fun: Callable[[float], float]) -> float:
  """Scores all possible shifts of a function."""
  best_score = np.inf
  best_lhs_max_value = np.inf
  best_rhs = np.inf
  for additive_const in np.arange(-1.0, 1.0, 0.02):

    def fun_shifted(x):
      return fun(x) + additive_const if fun(x) > 0 and np.abs(x) < 1 else 0  # pylint: disable=cell-var-from-loop

    score, lhs_max_value, rhs = score_function(fun_shifted)
    if score < best_score:
      best_score = score
      best_lhs_max_value = lhs_max_value
      best_rhs = rhs
  return best_score, best_lhs_max_value, best_rhs


def evaluate(hypers: Mapping[str, Any]) -> Mapping[str, float | None]:
  """Returns the numerical bound for the polygons if valid, or 0 if invalid."""
  result = {}
  feedback = {}
  del hypers
  fun_list, fun_string_list = list_of_functions()
  scores = [score_all_function_shifts(f) for f in fun_list]
  score_min = np.min([score[0] for score in scores])
  result['score_min'] = -score_min
  for i, score in enumerate(scores):
    if isinstance(score[1], float) and isinstance(score[2], float):
      feedback[
          f'\nThe LHS and RHS values of the {i}-th function'
          f' {fun_string_list[i]} were:'
      ] = f'{score[1]:.2f}, {score[2]:.2f}\n'
    else:
      feedback[
          f'\nThe LHS and RHS values of the {i}-th function'
          f' {fun_string_list[i]} were:'
      ] = f'{score[1]}, {score[2]}\n'

    feedback[
        f'The LHS/RHS values of this {i}-th function was (smaller is better):'
    ] = '\n'.join([
        f'{fun_string_list[i]}: {score[0]:.3f} '
        if score[0] <= 1000
        else 'too big to calculate'
    ])
  return result, feedback

In [ ]:
#@title Initial program

import itertools
import logging
import time
from scipy import integrate
import numpy as np
from scipy import optimize
import warnings
from typing import Any, Callable, Mapping

def list_of_functions() -> float:
  """Returns a list of functions to try."""
  fun1 = lambda x: x**2 + 1.0 / (np.abs(x) + 1)
  fun1_str = 'x**2 + 1 / (|x| + 1)'
  fun2 = lambda x: x**2 + 1.0 / (np.abs(x) + 2)
  fun2_str = 'x**2 + 1 / (|x| + 2)'
  fun3 = lambda x: x**2 + 1.0 / (np.abs(x) + 3)
  fun3_str = 'x**2 + 1 / (|x| + 3)'
  fun4 = lambda x: 1 / (x**2 + 5) + 1.0 / (np.abs(x) + 4)
  fun4_str = '1 / (x**2 + 5) + 1 / (|x| + 4)'
  fun5 = lambda x: 1.0 / (x**2 + 3) + 1.0 / (x**4 + 5)
  fun5_str = '1 / (x**2 + 3) + 1 / (x**4 + 5)'
  fun6 = lambda x: 1.0 / (x**2 + 1) + 1.0 / (np.abs(x) + 6)
  fun6_str = '1 / (x**2 + 1) + 1 / (|x| + 6)'

  def fun7(x):
    if x < -0.5:
      return 0
    if x > 0.5:
      return 0
    return np.sqrt(x + 2)

  fun7_str = 'sqrt(x + 2) if x > -0.5 and x < 0.5 else 0'

  def fun8(x):
    if x < -0.2:
      return 0
    if x > 0.2:
      return 0
    return 1 / np.sqrt(3 * x + 0.1)

  fun8_str = '1 / sqrt(3*x + 0.1) if x > -0.2 and x < 0.2 else 0'

  return [fun1, fun2, fun3, fun4, fun5, fun6, fun7, fun8], [
      fun1_str,
      fun2_str,
      fun3_str,
      fun4_str,
      fun5_str,
      fun6_str,
      fun7_str,
      fun8_str,
  ]


**Prompt used**

Act as an expert software developer and inequality specialist specializing in creating functions with certain properties. Your task is to find a function f that has the smallest possible value of the fraction LHS / RHS, where
LHS is the maximum possible value of the integral of f(t − x)f(x) dx from -inf to +inf, where t ranges from -1/2 to +1/2, and
RHS = (integral of f(x) dx from x = -1/4 to +1/4)*2

You can suggest a list of functions to try, as many as you want. You will see the values of LHS and RHS after you post your suggestions, and you can learn from your mistakes. You should give each function a short description or a name as well -- this way we can give feedback on them, and you will be able to identify which function we were talking about easier.

Always adhere to best practices in Python coding.
Do not use the random module. Your function always has to return a list of functions from R to R.

Before each program code, you will see some expert advice, as well as the score of the previous program. This advice is invaluable -- this is the only thing you have to guide you towards better and better functions. Make sure you make good use of it. Start your reply by commenting on your opinion of this advice. In each iteration, it's probably a good idea to throw out those functions that didn't perform well and replace them with new guesses.

## What AlphaEvolve found

This was the very first problem attempted in the project, before the search mode of AlphaEvolve had been developed. AlphaEvolve was asked to suggest mathematical functions directly (rather than bounded step functions), and the evaluation also tried thousands of simple transformations of each suggested function. The results highlighted the team's inexperience: since AlphaEvolve was allowed to suggest arbitrary functions, it eventually learned to exploit the numerical integration methods in the scoring function, producing highly irregular functions with impossibly high scores. The approach did not yield valid improvements to the known bounds $0.37 \leq C \leq 0.411$. The paper notes that if this problem were attempted again, the search mode in the space of bounded step functions with fixed step sizes would likely work much better, as that setup successfully improved bounds on all the other autocorrelation problems in the paper.